<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/20_rag_output_postprocessing/rag_output_postprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers scikit-learn --quiet

In [ ]:
raw_outputs = [
    "The capital of France is Paris.",
    "Paris is the capital of France. The question asks about France.",
    "Answer: Paris",
    "Paris."
]

In [ ]:
import re

def clean_answer(text):
    # Remove "Answer:" prefix
    text = re.sub(r"(?i)answer[:\-]*", "", text)

    # Remove extra sentences (keep first)
    text = text.split(".")[0]

    # Remove extra spaces
    text = text.strip()

    return text

In [8]:
for output in raw_outputs:
    cleaned = clean_answer(output)

    print("RAW:", output)
    print("CLEANED:", cleaned)
    print("-" * 40)

RAW: The capital of France is Paris.
CLEANED: The capital of France is Paris
----------------------------------------
RAW: Paris is the capital of France. The question asks about France.
CLEANED: Paris is the capital of France
----------------------------------------
RAW: Answer: Paris
CLEANED: Paris
----------------------------------------
RAW: Paris.
CLEANED: Paris
----------------------------------------


In [ ]:
# -------------------------------
# BASIC RAG PIPELINE (MINIMAL)
# -------------------------------

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load models
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Data
documents = [
    "Paris is the capital of France.",
    "Berlin is the capital of Germany.",
    "France is located in Europe."
]

# TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

# Embeddings
embeddings = embed_model.encode(documents)


def rag_pipeline(query):
    query_embedding = embed_model.encode([query])
    query_tfidf = vectorizer.transform([query])

    semantic_scores = cosine_similarity(query_embedding, embeddings)[0]
    keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]

    hybrid_scores = 0.5 * semantic_scores + 0.5 * keyword_scores

    best_index = hybrid_scores.argmax()
    context = documents[best_index]

    prompt = f"""
Answer the question using the context.

Context: {context}

Question: {query}
"""

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=30)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def rag_pipeline_clean(query):
    answer = rag_pipeline(query)
    return clean_answer(answer)

In [9]:
print(rag_pipeline_clean("What is the capital of France?"))

Paris
